In [1]:
#pip install akshare
!pip install baostock pandas 

In [2]:
# 导入 AkShare 财经数据接口库
# 用于获取 A 股、基金、指数、宏观经济等金融市场数据
import akshare as ak

# 导入 pandas 数据处理库
# 用于进行数据清洗、DataFrame 操作、数据分析等
import pandas as pd

# 输出当前安装的 AkShare 版本号
# 用于确认 AkShare 环境是否正确安装，以及记录运行环境版本
print(ak.__version__)

1.18.81


In [3]:
'''# 获取A股实时列表
stock_list = ak.stock_zh_a_spot_em()

stock_list.head()'''

'# 获取A股实时列表\nstock_list = ak.stock_zh_a_spot_em()\n\nstock_list.head()'

In [4]:
#stocks = stock_list["代码"].tolist()
#print(len(stocks))

In [5]:
# ==========================================
# 获取股票池
# 使用沪深300、中证500、中证1000作为A股样本池
# ==========================================

# 获取沪深300指数成分股
# symbol="000300" 对应沪深300指数
# 返回包含成分股代码、名称等信息的DataFrame
hs300 = ak.index_stock_cons_csindex(symbol="000300")

# 获取中证500指数成分股
# symbol="000905" 对应中证500指数
zz500 = ak.index_stock_cons_csindex(symbol="000905")

# 获取中证1000指数成分股
# symbol="000852" 对应中证1000指数
zz1000 = ak.index_stock_cons_csindex(symbol="000852")



#hs300 = ak.index_stock_cons(symbol="000300")
#zz500 = ak.index_stock_cons(symbol="000905")
#zz1000 = ak.index_stock_cons(symbol="000852")


# ==========================================
# 提取股票代码
# 将DataFrame中的股票代码字段转换为Python列表
# ==========================================

stocks_hs300 = hs300["成分券代码"].tolist()
stocks_zz500 = zz500["成分券代码"].tolist()
stocks_zz1000 = zz1000["成分券代码"].tolist()


# ==========================================
# 合并股票池
# 将三个指数成分股合并，构建覆盖大中小市值股票的训练样本池
# ==========================================

stocks_pool = (
    stocks_hs300
    + stocks_zz500
    + stocks_zz1000
)


# ==========================================
# 去除重复股票
# 部分股票可能同时属于多个指数，因此使用 set 去重
# 再转换回 list 方便后续循环下载数据
# ==========================================

stocks_pool = list(set(stocks_pool))


# 输出股票池规模
# 输出前10只股票代码检查数据是否正常
print("股票数量:", len(stocks_pool))
print(stocks_pool[:10])

股票数量: 1800
['600285', '002608', '688819', '000001', '300702', '002993', '300776', '300660', '600776', '688232']


In [6]:
import os
import time
from datetime import datetime
import shutil
import baostock as bs
import pandas as pd

# 设置路径
save_path = "./A_stock_raw"
#shutil.rmtree(r"C:\Users\schna\A股\A_stock_raw")  
if not os.path.exists(save_path):
    os.makedirs(save_path)

# 登录baostock
lg = bs.login()
print('baostock login:', lg.error_msg)

# 设定日期
start_date = "2015-01-01"
# 动态设置为昨天
end_date = (datetime.now() - pd.Timedelta(days=1)).strftime("%Y-%m-%d")

# 循环下载
for i, code in enumerate(stocks_pool):
    file = f"{save_path}/{code}.csv"
    if os.path.exists(file):
        print(code, "already exists")
        continue

    print(f"{i+1}/{len(stocks_pool)} downloading {code}")

    # 确定交易所前缀
    if code.startswith('6'):
        bs_code = f"sh.{code}"
    else:
        bs_code = f"sz.{code}"

    try:
        rs = bs.query_history_k_data_plus(
            bs_code,
            "date,open,high,low,close,volume,amount",
            start_date=start_date,
            end_date=end_date,
            frequency="d",
            adjustflag="2"   
        )

        # 转换为DataFrame
        data_list = []
        while (rs.error_code == '0') & rs.next():
            data_list.append(rs.get_row_data())
        df = pd.DataFrame(data_list, columns=rs.fields)

        if len(df) == 0:
            print(f"  {code} no data, skip")
            continue

        # 转换数据类型
        df = df.astype({
            'open': 'float',
            'high': 'float',
            'low': 'float',
            'close': 'float',
            'volume': 'int',
            'amount': 'float'
        })

        # 添加股票代码字段
        df['code'] = code

        # 保存
        df.to_csv(file, index=False, encoding='utf-8-sig')
        
    except Exception as e:
        print(f"  {code} failed: {e}")

# 登出
bs.logout()
print("All done!")

login success!
baostock login: success
600285 already exists
002608 already exists
688819 already exists
000001 already exists
300702 already exists
002993 already exists
300776 already exists
300660 already exists
600776 already exists
688232 already exists
002332 already exists
603927 already exists
600346 already exists
600746 already exists
688563 already exists
16/1800 downloading 600970
  600970 failed: invalid literal for int() with base 10: '': Error while type casting for column 'volume'
002402 already exists
600435 already exists
000902 already exists
601607 already exists
600879 already exists
002705 already exists
688182 already exists
601222 already exists
002487 already exists
000762 already exists
601000 already exists
301217 already exists
002101 already exists
603260 already exists
000917 already exists
600019 already exists
33/1800 downloading 002506
  002506 failed: invalid literal for int() with base 10: '': Error while type casting for column 'volume'
002938 alread

In [7]:
# ==========================================
# 合并所有股票历史行情数据
# 将单股票CSV文件整合为统一训练数据集
# ==========================================

import pandas as pd
import glob


# 获取所有股票CSV文件路径
# 每个CSV文件对应一只股票的历史行情数据
files = glob.glob("./A_stock_raw/*.csv")
# ==========================================
# 读取并合并所有股票数据
#
# pd.read_csv:
#     读取单个股票CSV文件
#
# pd.concat:
#     将多个DataFrame纵向拼接
#
# ignore_index=True:
#     重新生成连续索引
# ==========================================
df = pd.concat(
    [pd.read_csv(file) for file in files],
    ignore_index=True
)
# 查看数据规模
# shape返回(样本数量, 特征数量)
# head查看前几行数据，检查读取是否正常

print(df.shape)
print(df.head())

# ==========================================
# 日期格式转换
#
# 原始date通常为字符串格式
# 转换为datetime格式方便时间排序、
# 时间窗口计算以及技术指标生成
# ==========================================

df["date"] = pd.to_datetime(df["date"])
# ==========================================
# 数据排序
#
# 按股票代码和交易日期排序
# 保证每只股票的历史行情按照时间顺序排列
#
# 后续计算:
# - MA移动平均
# - RSI指标
# - 收益率
# - 滚动窗口特征
# 都需要基于时间顺序
# ==========================================
df = df.sort_values(
    ["code", "date"]
)

(3454769, 8)
         date      open      high       low     close     volume  \
0  2015-01-05  8.173241  8.321473  7.973893  8.188575  286043648   
1  2015-01-06  8.101680  8.377699  7.948336  8.065900  216642140   
2  2015-01-07  7.953447  8.091457  7.820549  7.912556  170012067   
3  2015-01-08  7.922778  7.958559  7.616090  7.646759  140771421   
4  2015-01-09  7.616090  8.111903  7.518972  7.708097  250850023   

         amount  code  
0  4.565388e+09     1  
1  3.453446e+09     1  
2  2.634796e+09     1  
3  2.128003e+09     1  
4  3.835378e+09     1  


In [8]:
import numpy as np
import pandas as pd
# ==========================================
# 数据排序
#
# 保证每只股票按照交易日期升序排列
# 后续:
# - shift()
# - rolling()
# - 技术指标计算
# 均依赖正确时间顺序
# ==========================================

df = df.sort_values(
    ["code", "date"]
)
# ==========================================
# Label构造
#
# 目标:
# 预测未来30个交易日股票是否上涨超过10%
#
# label:
#     1 -> 未来30日收益率 > 10%
#     0 -> 否
#
# 使用shift(-30):
#     获取未来第30天收盘价
#
# 注意:
#     不能使用未来信息作为feature，
#     否则会产生data leakage
# ==========================================
df["future_return_30"] = (
    df.groupby("code")["close"]
      .shift(-30)
    /
    df["close"]
    - 1
)


df["label"] = (
    df["future_return_30"] > 0.05
).astype(int)

# ==========================================
# Moving Average Features
#
# MA:
#     移动平均线，用于描述股票趋势
#
# MA5:
#     短期趋势
#
# MA20:
#     中期趋势
# ==========================================


df["MA5"] = (
    df.groupby("code")["close"]
      .transform(
          lambda x:x.rolling(5).mean()
      )
)


df["MA20"] = (
    df.groupby("code")["close"]
      .transform(
          lambda x:x.rolling(20).mean()
      )
)

# 当前价格相对于均线的位置
# 大于1表示价格高于均线

df["close_MA5_ratio"] = (
    df["close"]
    /
    df["MA5"].replace(0,np.nan)
)


df["close_MA20_ratio"] = (
    df["close"]
    /
    df["MA20"].replace(0,np.nan)
)

# MA5 / MA20
# 用于衡量短期趋势相对于长期趋势

df["MA5_MA20_ratio"] = (
    df["MA5"]
    /
    df["MA20"].replace(0,np.nan)
)

# ==========================================
# Bollinger Bands Features
#
# BOLL由:
# 中轨 + 上轨 + 下轨组成
#
# 用于衡量价格相对于历史波动区间的位置
# ==========================================


group_close = (
    df.groupby("code")["close"]
)


# 20日移动平均作为BOLL中轨

df["boll_mid"] = (
    group_close
    .transform(
        lambda x:x.rolling(20).mean()
    )
)

# 20日价格标准差

boll_std = (
    group_close
    .transform(
        lambda x:x.rolling(20).std()
    )
)

# 上轨 = 中轨 + 2倍标准差

df["boll_upper"] = (
    df["boll_mid"] 
    + 
    2*boll_std
)

# 下轨 = 中轨 - 2倍标准差

df["boll_lower"] = (
    df["boll_mid"] 
    -
    2*boll_std
)

# 当前价格处于BOLL区间的位置
# 0附近: 接近下轨
# 1附近: 接近上轨

df["boll_position"] = (
    (df["close"]-df["boll_lower"])
    /
    (df["boll_upper"]-df["boll_lower"])
    .replace(0,np.nan)
)

# ==========================================
# KDJ Features
#
# 添加:
# K
# D
# J
#
# 用于捕捉短期价格动量和超买超卖状态

def calculate_kdj(df, n=9):

    df = df.copy()

    # 每只股票分别计算
    def calculate(group):

        low_min = (
            group["low"]
            .rolling(n)
            .min()
        )

        high_max = (
            group["high"]
            .rolling(n)
            .max()
        )

        # RSV指标
        rsv = (
            (group["close"] - low_min)
            /
            (high_max - low_min)
        )

        # K值
        group["K"] = (
            rsv
            .ewm(
                com=2,
                adjust=False
            )
            .mean()
        )


        # D值
        group["D"] = (
            group["K"]
            .ewm(
                com=2,
                adjust=False
            )
            .mean()
        )


        # J值
        group["J"] = (
            3 * group["K"]
            -
            2 * group["D"]
        )

        return group


    df = (
        df
        .groupby(
            "code",
            group_keys=False
        )
        .apply(calculate)
    )

    return df

# ==========================================

df = calculate_kdj(
    df,
    n=9
)

# K-D差值
# 衡量K线和D线之间的趋势强弱

df["K_D_diff"] = (
    df["K"]
    -
    df["D"]
)

# ==========================================
# Volume Features
#
# volume_ratio:
# 当前成交量 / 20日平均成交量
#
# 用于判断成交量是否异常放大
# ==========================================


volume_ma20 = (
    df.groupby("code")["volume"]
      .transform(
          lambda x:x.rolling(20).mean()
      )
)


df["volume_ratio"] = (
    df["volume"]
    /
    volume_ma20.replace(0,np.nan)
)

# ==========================================
# Momentum Features
#
# 衡量股票近期涨跌趋势
# ==========================================


def safe_return(x, period):

    return (
        x
        /
        x.shift(period).replace(0,np.nan)
        -
        1
    )

# 5日收益率

df["return_5"] = (
    df.groupby("code")["close"]
      .transform(
          lambda x:safe_return(x,5)
      )
)

# 20日收益率

df["return_20"] = (
    df.groupby("code")["close"]
      .transform(
          lambda x:safe_return(x,20)
      )
)

# ==========================================
# K线形态 Features
#
# 捕捉单日价格行为
# ==========================================


# 实体长度比例

df["body"] = (
    df["close"]-df["open"]
) / df["open"]

# 上影线比例

df["upper_shadow"] = (
    df["high"]
    -
    df[["open","close"]].max(axis=1)
) / df["open"]

# 下影线比例

df["lower_shadow"] = (
    df[["open","close"]].min(axis=1)
    -
    df["low"]
) / df["open"]

# ==========================================
# Volatility Feature
#
# volatility_20:
# 20日收益率标准差
#
# 衡量股票近期风险和波动水平
# ==========================================


daily_return = (
    df.groupby("code")["close"]
      .pct_change()
)


df["volatility_20"] = (
    daily_return
    .groupby(df["code"])
    .transform(
        lambda x:x.rolling(20).std()
    )
)

# ==========================================
# 数据清洗
#
# 处理:
# - 无穷值
# - 缺失值
#
# 缺失值主要来自:
# - rolling窗口不足
# - 除零错误
# ==========================================


df = df.replace(
    [np.inf,-np.inf],
    np.nan
)

# ==========================================
# 定义模型输入特征
# ==========================================

features = [

    # MA趋势指标
    "close_MA5_ratio",
    "close_MA20_ratio",
    "MA5_MA20_ratio",

    # BOLL指标
    "boll_position",

    # KDJ指标
    "K",
    "D",
    "J",
    "K_D_diff",

    # 成交量指标
    "volume_ratio",

    # 动量指标
    "return_5",
    "return_20",

    # K线形态
    "body",
    "upper_shadow",
    "lower_shadow",

    # 波动率
    "volatility_20"
]

# 删除无法用于训练的数据
# 保证所有feature和label均有效

df = df.dropna(
    subset=features + ["label"]
)

# 输出最终训练数据规模
print(df.shape)


# 检查剩余缺失值

print(df[features].isna().sum())

C:\Users\schna\AppData\Local\Temp\ipykernel_19216\3612610212.py:235: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calculate)


(3385224, 30)
close_MA5_ratio     0
close_MA20_ratio    0
MA5_MA20_ratio      0
boll_position       0
K                   0
D                   0
J                   0
K_D_diff            0
volume_ratio        0
return_5            0
return_20           0
body                0
upper_shadow        0
lower_shadow        0
volatility_20       0
dtype: int64


In [9]:
import numpy as np
# ==========================================
# Feature Data Quality Check
#
# 检查机器学习输入特征的数据质量
#
# 检查内容:
# 1. NaN数量
#    - 检查缺失值情况
#
# 2. Inf数量
#    - 检查正无穷和负无穷
#    - 通常由除零操作产生
#
# 3. 最大值/最小值
#    - 检查异常极端值
#    - 判断是否存在数据异常
#
# 目的:
# 在模型训练前确保所有feature数据有效，
# 避免异常数据影响XGBoost训练结果
# ==========================================


# 遍历所有模型输入特征

for col in features:

    print(
        col,

        # 缺失值数量
        "NaN:",
        df[col].isna().sum(),

        # 无穷值数量
        # 包括:
        # +inf
        # -inf
        "Inf:",
        np.isinf(df[col]).sum(),

        # 特征最大值
        # 用于发现异常高值

        "Max:",
        df[col].max(),

        # 特征最小值
        # 用于发现异常低值

        "Min:",
        df[col].min()
    )

close_MA5_ratio NaN: 0 Inf: 0 Max: 2.518944709514454 Min: 0.7140603164585494
close_MA20_ratio NaN: 0 Inf: 0 Max: 3.5220249190621007 Min: 0.4555528852572832
MA5_MA20_ratio NaN: 0 Inf: 0 Max: 2.286378909278883 Min: 0.5267862325149456
boll_position NaN: 0 Inf: 0 Max: 1.5793859744011265 Min: -0.5621322907597848
K NaN: 0 Inf: 0 Max: 1.0 Min: 2.636335952001687e-38
D NaN: 0 Inf: 0 Max: 1.0 Min: 9.355158769976366e-12
J NaN: 0 Inf: 0 Max: 2.3333333333333335 Min: -1.3333333332955655
K_D_diff NaN: 0 Inf: 0 Max: 0.6666666666666667 Min: -0.6666666666558412
volume_ratio NaN: 0 Inf: 0 Max: 20.0 Min: 0.0
return_5 NaN: 0 Inf: 0 Max: 3.2217194570135748 Min: -0.41323106423777567
return_20 NaN: 0 Inf: 0 Max: 5.733412322274881 Min: -0.7378939662309368
body NaN: 0 Inf: 0 Max: 0.3619897269532307 Min: -0.2585
upper_shadow NaN: 0 Inf: 0 Max: 0.2966056513239737 Min: 0.0
lower_shadow NaN: 0 Inf: 0 Max: 0.21256544502617797 Min: 0.0
volatility_20 NaN: 0 Inf: 0 Max: 0.6858079436391877 Min: 5.380058273176431e-08


In [18]:
# ==========================================
# 定义机器学习输入特征
#
# features:
#     用于训练模型的变量集合
#
# 包含:
# - MA趋势指标
# - BOLL波动指标
# - KDJ动量指标
# - 成交量指标
# - 收益率动量指标
#
# 这些特征将作为XGBoost模型的输入变量(X)
# ==========================================

features = [

    # MA趋势指标
    "close_MA5_ratio",
    "close_MA20_ratio",
    "MA5_MA20_ratio",

    # BOLL指标
    "boll_position",

    # KDJ指标
    "K",
    "D",
    "J",
    "K_D_diff",

    # 成交量指标
    "volume_ratio",

    # 动量指标
    "return_5",
    "return_20",

    # K线形态
    "body",
    "upper_shadow",
    "lower_shadow",

    # 波动率
    "volatility_20"
]

# ==========================================
# 删除缺失数据
#
# 删除包含NaN值的样本
#
# NaN来源:
# - rolling窗口初始阶段无法计算指标
# - shift计算未来收益导致最后若干天无Label
# - 技术指标计算产生缺失值
#
# 目的:
# 保证输入模型的数据完整，
# 避免训练过程中出现异常
# ==========================================

df = df.dropna()

In [19]:
X = df[features]
y = df["label"]

In [20]:
# ==========================================
# Train/Test Split
#
# 按时间顺序划分数据集
#
# 前80%的历史数据:
#     作为训练集
#
# 后20%的最新数据:
#     作为测试集
#
# 金融时间序列不能随机打乱划分，
# 避免未来数据泄露(Data Leakage)
# ==========================================


# 确保按照时间排序
df = df.sort_values(
    ["date"]
)


# 获取80%位置作为训练集截止点

split_index = int(
    len(df) * 0.8
)


# 前80%数据训练

train = df.iloc[:split_index]


# 后20%数据测试

test = df.iloc[split_index:]

X_train = train[features]
y_train = train["label"]

X_test = test[features]
y_test = test["label"]

In [21]:
from xgboost import XGBClassifier
# ==========================================
# 初始化XGBoost分类模型
#
# XGBoost:
#     基于梯度提升树(Gradient Boosting Decision Tree)
#     的机器学习算法
#
# 用途:
#     根据技术指标预测股票未来上涨概率
#
# 参数说明:
#
# n_estimators:
#     弱学习器数量（决策树数量）
#     越大模型复杂度越高
#
# max_depth:
#     单棵树最大深度
#     控制模型复杂程度，防止过拟合
#
# learning_rate:
#     学习率
#     控制每棵树对最终结果的贡献
#
# subsample:
#     每棵树随机采样训练数据比例
#     用于增强模型泛化能力
#
# colsample_bytree:
#     每棵树随机采样特征比例
#     降低特征相关性导致的过拟合
#
# tree_method="hist":
#     使用Histogram算法加速训练
#     适合大规模金融数据
#
# eval_metric="logloss":
#     使用Log Loss作为评价指标
#     衡量分类概率预测质量
# ==========================================


model = XGBClassifier(
    n_estimators=1000,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="logloss"
)

# ==========================================
# 模型训练
#
# X_train:
#     训练集特征
#
# y_train:
#     训练集标签
#
# eval_set:
#     使用测试集监控模型训练过程
#
# verbose=100:
#     每100轮输出一次训练状态
#
# 注意:
#     金融时间序列必须按照时间顺序训练，
#     避免未来数据泄露
# ==========================================


model.fit(
    X_train,
    y_train,
    eval_set=[
        (X_test,y_test)
    ],
    verbose=100
)

# ==========================================
# Feature Importance Analysis
#
# 分析不同特征对于模型预测的重要程度
#
# feature_importances_:
#     XGBoost基于树分裂贡献计算特征重要性
#
# 用途:
#     判断哪些技术指标对股票预测贡献最大
# ==========================================


import pandas as pd


importance = pd.DataFrame({

    # 特征名称
    "feature": features,

    # 模型计算出的重要性
    "importance": model.feature_importances_

})


# 按重要性降序排列
# 排名前面的指标对模型贡献更大

importance.sort_values(
    "importance",
    ascending=False
)

[0]	validation_0-logloss:0.67714
[100]	validation_0-logloss:0.67686
[200]	validation_0-logloss:0.67807
[300]	validation_0-logloss:0.67897
[400]	validation_0-logloss:0.67952
[500]	validation_0-logloss:0.67999
[600]	validation_0-logloss:0.68030
[700]	validation_0-logloss:0.68061
[800]	validation_0-logloss:0.68084
[900]	validation_0-logloss:0.68111
[999]	validation_0-logloss:0.68133


,feature,importance
10,return_20,0.158671
14,volatility_20,0.123275
2,MA5_MA20_ratio,0.102365
1,close_MA20_ratio,0.085300
5,D,0.058748
9,return_5,0.058561
8,volume_ratio,0.056151
0,close_MA5_ratio,0.052455
12,upper_shadow,0.048594
6,J,0.047523


In [22]:
from sklearn.metrics import classification_report


# ==========================================
# Model Evaluation
#
# 使用训练好的XGBoost模型对测试集进行预测
#
# pred:
#     模型输出的分类结果
#
# 0:
#     股票未来没有达到目标涨幅
#
# 1:
#     股票未来达到目标涨幅
# ==========================================

pred = model.predict(
    X_test
)
# ==========================================
# Classification Report
#
# 输出分类模型评价指标
#
# 包含:
#
# precision:
#     精确率
#     预测为上涨的股票中，实际上涨的比例
#
# recall:
#     召回率
#     所有实际上涨股票中，被模型发现的比例
#
# f1-score:
#     precision和recall的综合指标
#
# support:
#     每个类别的样本数量
#
# accuracy:
#     总体预测正确比例
# ==========================================
print(
    classification_report(
        y_test,
        pred
    )
)


model.save_model(
    "xgb_model.json"
)

              precision    recall  f1-score   support

           0       0.61      1.00      0.76    399560
           1       0.58      0.01      0.02    252749

    accuracy                           0.61    652309
   macro avg       0.59      0.50      0.39    652309
weighted avg       0.60      0.61      0.47    652309



In [23]:
import baostock as bs
import pandas as pd
import numpy as np


from xgboost import XGBClassifier
import pandas as pd



# ==========================================
# 2. 加载已经训练好的模型
# ==========================================

model = XGBClassifier()


model.load_model(
    "xgb_model.json"
)


df["future_return_30"] = (
    df.groupby("code")["close"]
      .shift(-30)
    /
    df["close"]
    - 1
)


df["label"] = (
    df["future_return_30"] > 0.05
).astype(int)

# ==========================================
# Moving Average Features
#
# MA:
#     移动平均线，用于描述股票趋势
#
# MA5:
#     短期趋势
#
# MA20:
#     中期趋势
# ==========================================


df["MA5"] = (
    df.groupby("code")["close"]
      .transform(
          lambda x:x.rolling(5).mean()
      )
)


df["MA20"] = (
    df.groupby("code")["close"]
      .transform(
          lambda x:x.rolling(20).mean()
      )
)

# 当前价格相对于均线的位置
# 大于1表示价格高于均线

df["close_MA5_ratio"] = (
    df["close"]
    /
    df["MA5"].replace(0,np.nan)
)


df["close_MA20_ratio"] = (
    df["close"]
    /
    df["MA20"].replace(0,np.nan)
)

# MA5 / MA20
# 用于衡量短期趋势相对于长期趋势

df["MA5_MA20_ratio"] = (
    df["MA5"]
    /
    df["MA20"].replace(0,np.nan)
)

# ==========================================
# Bollinger Bands Features
#
# BOLL由:
# 中轨 + 上轨 + 下轨组成
#
# 用于衡量价格相对于历史波动区间的位置
# ==========================================


group_close = (
    df.groupby("code")["close"]
)


# 20日移动平均作为BOLL中轨

df["boll_mid"] = (
    group_close
    .transform(
        lambda x:x.rolling(20).mean()
    )
)

# 20日价格标准差

boll_std = (
    group_close
    .transform(
        lambda x:x.rolling(20).std()
    )
)

# 上轨 = 中轨 + 2倍标准差

df["boll_upper"] = (
    df["boll_mid"] 
    + 
    2*boll_std
)

# 下轨 = 中轨 - 2倍标准差

df["boll_lower"] = (
    df["boll_mid"] 
    -
    2*boll_std
)

# 当前价格处于BOLL区间的位置
# 0附近: 接近下轨
# 1附近: 接近上轨

df["boll_position"] = (
    (df["close"]-df["boll_lower"])
    /
    (df["boll_upper"]-df["boll_lower"])
    .replace(0,np.nan)
)

# ==========================================
# KDJ Features
#
# 添加:
# K
# D
# J
#
# 用于捕捉短期价格动量和超买超卖状态

def calculate_kdj(df, n=9):

    df = df.copy()

    # 每只股票分别计算
    def calculate(group):

        low_min = (
            group["low"]
            .rolling(n)
            .min()
        )

        high_max = (
            group["high"]
            .rolling(n)
            .max()
        )

        # RSV指标
        rsv = (
            (group["close"] - low_min)
            /
            (high_max - low_min)
        )

        # K值
        group["K"] = (
            rsv
            .ewm(
                com=2,
                adjust=False
            )
            .mean()
        )


        # D值
        group["D"] = (
            group["K"]
            .ewm(
                com=2,
                adjust=False
            )
            .mean()
        )


        # J值
        group["J"] = (
            3 * group["K"]
            -
            2 * group["D"]
        )

        return group


    df = (
        df
        .groupby(
            "code",
            group_keys=False
        )
        .apply(calculate)
    )

    return df

# ==========================================

df = calculate_kdj(
    df,
    n=9
)

# K-D差值
# 衡量K线和D线之间的趋势强弱

df["K_D_diff"] = (
    df["K"]
    -
    df["D"]
)

# ==========================================
# Volume Features
#
# volume_ratio:
# 当前成交量 / 20日平均成交量
#
# 用于判断成交量是否异常放大
# ==========================================


volume_ma20 = (
    df.groupby("code")["volume"]
      .transform(
          lambda x:x.rolling(20).mean()
      )
)


df["volume_ratio"] = (
    df["volume"]
    /
    volume_ma20.replace(0,np.nan)
)

# ==========================================
# Momentum Features
#
# 衡量股票近期涨跌趋势
# ==========================================


def safe_return(x, period):

    return (
        x
        /
        x.shift(period).replace(0,np.nan)
        -
        1
    )

# 5日收益率

df["return_5"] = (
    df.groupby("code")["close"]
      .transform(
          lambda x:safe_return(x,5)
      )
)

# 20日收益率

df["return_20"] = (
    df.groupby("code")["close"]
      .transform(
          lambda x:safe_return(x,20)
      )
)

# ==========================================
# K线形态 Features
#
# 捕捉单日价格行为
# ==========================================


# 实体长度比例

df["body"] = (
    df["close"]-df["open"]
) / df["open"]

# 上影线比例

df["upper_shadow"] = (
    df["high"]
    -
    df[["open","close"]].max(axis=1)
) / df["open"]

# 下影线比例

df["lower_shadow"] = (
    df[["open","close"]].min(axis=1)
    -
    df["low"]
) / df["open"]

# ==========================================
# Volatility Feature
#
# volatility_20:
# 20日收益率标准差
#
# 衡量股票近期风险和波动水平
# ==========================================


daily_return = (
    df.groupby("code")["close"]
      .pct_change()
)


df["volatility_20"] = (
    daily_return
    .groupby(df["code"])
    .transform(
        lambda x:x.rolling(20).std()
    )
)

# ==========================================
# 数据清洗
#
# 处理:
# - 无穷值
# - 缺失值
#
# 缺失值主要来自:
# - rolling窗口不足
# - 除零错误
# ==========================================


df = df.replace(
    [np.inf,-np.inf],
    np.nan
)

# ==========================================
# 定义模型输入特征
# ==========================================

features = [

    # MA趋势指标
    "close_MA5_ratio",
    "close_MA20_ratio",
    "MA5_MA20_ratio",

    # BOLL指标
    "boll_position",

    # KDJ指标
    "K",
    "D",
    "J",
    "K_D_diff",

    # 成交量指标
    "volume_ratio",

    # 动量指标
    "return_5",
    "return_20",

    # K线形态
    "body",
    "upper_shadow",
    "lower_shadow",

    # 波动率
    "volatility_20"
]

# 删除无法用于训练的数据
# 保证所有feature和label均有效

df = df.dropna(
    subset=features + ["label"]
)




    # 返回最新一天

X_today = (
    df[features]
    .iloc[-1:]
)




X = get_stock_features("600519")





model.predict(X)
print("如果是1则30天后收盘价上涨5%及以上， 0 则30天后收盘价不会上涨5%以上")

C:\Users\schna\AppData\Local\Temp\ipykernel_19216\38217961.py:224: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calculate)


login success!
logout success!


array([0])